# State-classification workflow

This notebook covers both binary and ternary readout classification.
Start with a standard `ge` experiment for the 2-state classifier, then
move to `ge-ef-cr` mode when you want to resolve `|g>`, `|e>`, and `|f>`.

## 1. Build a 2-state classifier

Start from the ordinary `ge` configuration and collect the readout
clusters for `|g>` and `|e>`.

In [ ]:
import numpy as np

import qubex as qx

exp_ge = qx.Experiment(
    system_id="YOUR_SYSTEM_ID",
    muxes=[0],
    # config_dir="/path/to/qubex-config/config",
    # params_dir="/path/to/qubex-config/params/YOUR_SYSTEM_ID",
)

In [ ]:
Q0 = exp_ge.qubit_labels[0]

print("target qubit:", Q0)

## 2. Connect to the setup

Connect before you collect any state clusters.

In [ ]:
exp_ge.connect()

## 3. Check the waveform

Use a waveform check to confirm that the readout path is stable before classification.

In [ ]:
waveform_result = exp_ge.check_waveform()

## 4. Prepare the `ge` pulse set

Measure a baseline Rabi oscillation, calibrate the half-pi pulse, and save the note before building the 2-state classifier.

In [ ]:
rabi_result = exp_ge.obtain_rabi_params()
hpi_result = exp_ge.calibrate_hpi_pulse()
exp_ge.calib_note.save()

## 5. Measure the 2-state clusters

Collect the `|g>` and `|e>` readout distributions for the two-state classifier.

In [ ]:
binary_distributions = exp_ge.measure_state_distribution(
    exp_ge.qubit_labels,
    n_states=2,
)

## 6. Build the 2-state classifier

Fit the classifier from the measured two-state clusters.

In [ ]:
binary_classifier = exp_ge.build_classifier(
    exp_ge.qubit_labels,
    n_states=2,
    n_shots=10000,
)

## 7. Create an `Experiment` in `ge-ef-cr` mode

Open a second experiment in `ge-ef-cr` mode for three-state classification.

In [ ]:
exp_gef = qx.Experiment(
    system_id="YOUR_SYSTEM_ID",
    muxes=[0],
    configuration_mode="ge-ef-cr",
    # config_dir="/path/to/qubex-config/config",
    # params_dir="/path/to/qubex-config/params/YOUR_SYSTEM_ID",
)

## 8. Measure the EF Rabi oscillation

Use the EF Rabi scan to confirm that the `|f>` manifold can be prepared reliably.

In [ ]:
exp_gef.connect()
ef_rabi_result = exp_gef.obtain_ef_rabi_params(
    exp_gef.qubit_labels,
    time_range=np.arange(0, 201, 4),
    n_shots=1024,
)

## 9. Calibrate the EF half-pi pulse

Calibrate the EF half-pi pulse before collecting the three-state clusters.

In [ ]:
ef_hpi_result = exp_gef.calibrate_ef_hpi_pulse(
    exp_gef.qubit_labels,
    n_rotations=1,
)

## 10. Measure the 3-state clusters

Collect the `|g>`, `|e>`, and `|f>` distributions for ternary classification.

In [ ]:
ternary_distributions = exp_gef.measure_state_distribution(
    exp_gef.qubit_labels,
    n_states=3,
)

## 11. Build the 3-state classifier

Fit the ternary classifier from the measured three-state clusters.

In [ ]:
ternary_classifier = exp_gef.build_classifier(
    exp_gef.qubit_labels,
    n_states=3,
    n_shots=1000,
)